# factorio-planner — Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BernardRivera13/factorio-planner/blob/main/docs/tutorial.ipynb)

This notebook is a complete walkthrough of **factorio-planner**.

1. Compute any production chain in seconds
2. Model productivity and speed bonuses
3. Export plans as JSON
4. Visualise chains as graphs
5. Add custom / modded recipes

In [ ]:
# Install directly from PyPI
!pip install -q "factorio-planner[graph]"

## 1. Basic usage

How many stone furnaces do I need for **1 iron plate/second**?

In [ ]:
from factorio_planner import Calculator
from factorio_planner.formatter import format_plan

calc = Calculator()
plan = calc.compute("iron-plate", rate=1.0)
print(plan.summary())

In [ ]:
print(format_plan(plan))

## 2. Electronic circuits (10/s)

A classic mid-game goal. The calculator resolves every ingredient recursively.

In [ ]:
plan = calc.compute("electronic-circuit", rate=10)
print(format_plan(plan))

## 3. Science packs

Late-game: **1 utility science pack per second**.

In [ ]:
plan_usp = calc.compute("utility-science-pack", rate=1)
print(format_plan(plan_usp))

## 4. Productivity modules

`productivity_bonus=0.4` models fully-researched productivity (each machine outputs 40% more).

In [ ]:
calc_prod = Calculator(productivity_bonus=0.4)
plan_base = Calculator().compute("advanced-circuit", rate=5)
plan_prod = calc_prod.compute("advanced-circuit", rate=5)

for machine in plan_base.machine_totals:
    b = plan_base.machine_totals[machine]
    p = plan_prod.machine_totals.get(machine, 0)
    print(f"{machine}: {b:.2f} -> {p:.2f}  ({(p-b)/b*100:+.1f}%)")

## 5. Speed beacons

`speed_bonus=0.5` adds +50% throughput from beacons, reducing machine count.

In [ ]:
calc_fast = Calculator(speed_bonus=0.5)
plan_fast = calc_fast.compute("processing-unit", rate=1)
print(plan_fast.summary())

## 6. JSON output

In [ ]:
import json
from factorio_planner.formatter import to_dict

plan = calc.compute("electronic-circuit", rate=10)
d = to_dict(plan)
print(json.dumps(d, indent=2)[:600], "...")

## 7. Graph visualisation

Requires `factorio-planner[graph]` (already installed).

In [ ]:
from factorio_planner.graph import draw
import matplotlib.pyplot as plt

plan = calc.compute("electronic-circuit", rate=10)
fig = draw(plan, figsize=(12, 7))
plt.show()

In [ ]:
# Chemical science pack chain
plan_chem = calc.compute("chemical-science-pack", rate=1)
fig2 = draw(plan_chem, figsize=(14, 9))
plt.show()

## 8. Custom / modded recipes

In [ ]:
from factorio_planner import RecipeDB
from factorio_planner.models import Recipe

db = RecipeDB()
db.add(Recipe(
    name="quantum-processor",
    ingredients={"processing-unit": 5, "advanced-circuit": 10},
    products={"quantum-processor": 1},
    crafting_time=5.0,
    machine="assembling-machine-3",
))

calc_mod = Calculator(db=db)
plan_mod = calc_mod.compute("quantum-processor", rate=2)
print(format_plan(plan_mod))

## 9. CLI (works in Colab too)

In [ ]:
!factorio-planner electronic-circuit --rate 10

In [ ]:
!factorio-planner advanced-circuit --rate 5 --productivity 0.4 --no-tree

In [ ]:
!factorio-planner iron-plate --rate 3 --json

## Summary

| Feature | API | CLI |
|---|---|---|
| Basic plan | `calc.compute(item, rate)` | `factorio-planner ITEM --rate N` |
| Productivity | `Calculator(productivity_bonus=0.4)` | `--productivity 0.4` |
| Speed beacons | `Calculator(speed_bonus=0.5)` | `--speed 0.5` |
| JSON export | `to_dict(plan)` | `--json` |
| Graph | `draw(plan)` | — |
| Custom recipes | `db.add(recipe)` / `RecipeDB.from_json(path)` | — |